# K-means Clustering
## Using Snowpark Python and Scikit-Learn
### Overview
This notebook finds a set of cluster centers for customer addresses. 

Note that, for simplicity, the customer addresses are already resolved to geolocations (latitude and longitude). A production system could include a call to an external service that performs this geolocation on new customer addresses.

Steps:
- Setup
- Load and Explore Data
- Cluster to Find 3 Central Locations
- Save Cluster Centers to Snowflake
- Areas for Further Investigation/Work

### Setup

In [ ]:
# Install folium -- not included in this Python build
!pip install folium

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.session import Session

import folium

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

* Load configuration and connect to Snowflake

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

### Load and Explore Data

- Examine via a Snowpark DataFrame

In [ ]:
membersDF = session.table('data_science_db.raw.geo_members')

In [ ]:
membersDF.schema.fields

In [ ]:
membersDF.count()

In [ ]:
membersDF.show(5)

* For the purpose of modeling, we will keep only member_id, lat, and lon (latitude and longitude).

In [ ]:
membersDF = membersDF.select('member_id', 'lat', 'lon')
membersDF.show(5)

* Plot the members' home address locations.

In [ ]:
def plot_points(df):
    
    # Map centered on SFO airport
    m = folium.Map(location=[37.62, -122.365], zoom_start=7.5)
    
    # Obtain [lat, lon] columns as list of Python Row objects to plot
    pointsLST = df.select('lat', 'lon').collect()
    
    for point in pointsLST:
        folium.CircleMarker(location=[point['LAT'], point['LON']], \
                           radius=1) \
              .add_to(m)
    return(m)

In [ ]:
plot_points(membersDF)

Members are located in the vicinity of the San Francisco (SFO) airport.

### Cluster to Find 3 Central Locations
Given these customer home address locations, find three general locations that center around clusters of customers. To compete in this market, we will consider providing a free or inexpensive luxury shuttle from each of these "transportation hubs" to the SFO airport.

In [ ]:
# Fetch points to a Pandas DataFrame for training
pointsPDF = membersDF.select('lat', 'lon').toPandas()

* Choose K-means for a clustering algorithm, and train the model to find three cluster centers using the available data.

In [ ]:
from snowflake.ml.modeling.cluster import KMeans as KM
k = 3
kmeans = KM(n_clusters=k)

# Get an array of predictions
predictions = kmeans.fit_predict(pointsPDF)

# Convert to an array
predictions =predictions['OUTPUT_0'].to_numpy()

- Show the cluster centers

In [ ]:
# cluster_centers_ is not yet supported in Snowpark KMeans.
# So we revert back to sklearn:
kmeans.to_sklearn().cluster_centers_

- Score the data points with cluster membership, and show a few.

In [ ]:
import pandas as pd
predictionsPDF = pd.DataFrame(predictions, columns=['prediction'])
scored_pointsPDF = pointsPDF.join(predictionsPDF)
scored_pointsPDF.head()

- Plot the clusters and cluster centers.

In [ ]:
def plot_clusters(centers, clustered_pointsPDF):
    colors = ['orange', 'yellow', 'green', 'blue', 'purple']
    m = folium.Map(location=[37.62, -122.365], zoom_start=7.5)

    for index, point in clustered_pointsPDF.iterrows():
        folium.CircleMarker(location=[point['LAT'], point['LON']], 
                          color=colors[int(point['prediction'])],
                          radius=1) \
              .add_to(m)
        
    for (i, center) in enumerate(centers):
        folium.CircleMarker(location=center.tolist(),color='red') \
              .add_to(m)

    return(m)


In [ ]:
plot_clusters(kmeans.to_sklearn().cluster_centers_, scored_pointsPDF)

- How many customers are served by each hub?

In [ ]:
for i in range(k):
    n = scored_pointsPDF[scored_pointsPDF["prediction"]==i].shape[0]
    print (i, n)

### Save Cluster Centers to Snowflake

In [ ]:
# Create Pandas DataFrame to save
centersPDF = pd.DataFrame(kmeans.to_sklearn().cluster_centers_, columns=['LAT', 'LON'])
centersPDF.reset_index(inplace=True)
centersPDF = centersPDF.rename({'index': 'CLUSTER_ID'}, axis=1)

- Create Snowflake schema in which to save cluster centers

In [ ]:
session.sql('create or replace schema clustering').collect()

- Save as table **CLUSTER_CENTERS**.

In [ ]:
# Save to a Snowflake table
session.write_pandas(centersPDF, 'CLUSTER_CENTERS', schema='CLUSTERING', auto_create_table=True)

### Areas for Further Investigation/Work
* Find a model that groups the members into four clusters. Plot the results.
* Try five clusters.
* Can you think of additional questions to pursue?